# ComfyUI su GPU T4 (Google Colab)

Esegui le celle **in ordine**, dall'alto in basso.

1. **Configurazione**: scegli se salvare le immagini su Google Drive.
2. **Controllo GPU**: verifica di avere la T4.
3. **Installazione**: ComfyUI + ComfyUI-Manager.
4. **Google Drive** (opzionale).
5. **Modelli**: scegli quali scaricare.
6. **Avvio**: alla fine compare un link `https://....trycloudflare.com`. Aprilo nel browser: è l'interfaccia di ComfyUI.

Lascia l'ultima cella in esecuzione finché lavori. Se la interrompi, ComfyUI si spegne.

In [ ]:
# === 1. CONFIGURAZIONE ===

# True  = le immagini generate vengono salvate direttamente su Google Drive
#         (cartella MyDrive/ComfyUI_output) e non si perdono a fine sessione.
# False = le immagini restano sulla macchina Google e vengono cancellate
#         quando la sessione finisce (scaricale a mano dall'interfaccia).
SALVA_SU_DRIVE = True
CARTELLA_DRIVE = 'ComfyUI_output'

In [ ]:
# === 2. CONTROLLO GPU ===
# Deve comparire una 'Tesla T4'. Se dà errore, il runtime non ha la GPU:
# ricollega il kernel scegliendo una macchina con GPU T4.
!nvidia-smi

In [ ]:
# === 3. INSTALLAZIONE COMFYUI + MANAGER ===
import os

COMFY = '/content/ComfyUI'
MANAGER = f'{COMFY}/custom_nodes/ComfyUI-Manager'

if not os.path.exists(COMFY):
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY}
else:
    !git -C {COMFY} pull

if not os.path.exists(MANAGER):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager {MANAGER}

!pip install -q -r {COMFY}/requirements.txt
!pip install -q -r {MANAGER}/requirements.txt
print('Installazione completata.')

In [ ]:
# === 4. GOOGLE DRIVE (opzionale) ===
OUTPUT_DIR = f'{COMFY}/output'

if SALVA_SU_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_DIR = f'/content/drive/MyDrive/{CARTELLA_DRIVE}'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
    except Exception as e:
        print('Drive non montato:', e)

print('Le immagini verranno salvate in:', OUTPUT_DIR)

In [ ]:
# === 5. MODELLI ===
# Togli il # davanti ai modelli che vuoi scaricare.
# I modelli stanno sulla macchina Google: vanno riscaricati a ogni nuova sessione.
# Altri modelli li puoi installare anche da ComfyUI-Manager, dentro l'interfaccia.
MODELLI = {
    # Stable Diffusion 1.5 (~2 GB, veloce)
    # 'checkpoints/v1-5-pruned-emaonly-fp16.safetensors':
    #     'https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors',

    # SDXL base 1.0 (~7 GB, qualità migliore, più lento)
    # 'checkpoints/sd_xl_base_1.0.safetensors':
    #     'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',

    # Z-Image-Turbo (~21 GB in tutto, ottimo e veloce: 8 passi): personaggi del cartone,
    # workflow workflows/zimage_turbo_t2i.json e zimage_turbo_img2img.json
    'diffusion_models/z_image_turbo_bf16.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/diffusion_models/z_image_turbo_bf16.safetensors',
    'text_encoders/qwen_3_4b.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors',
    'vae/ae.safetensors':
        'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors',

    # Wan 2.2 TI2V 5B (~18 GB in tutto): anima le scene (immagine -> video)
    # Workflow: workflows/wan22_5b_i2v.json
    'diffusion_models/wan2.2_ti2v_5B_fp16.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',
    'text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'vae/wan2.2_vae.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors',

    # VAE per SDXL
    # 'vae/sdxl_vae.safetensors':
    #     'https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors',
}

# wget -c riprende i download interrotti e salta quelli già completi.
for percorso, url in MODELLI.items():
    dest = f'{COMFY}/models/{percorso}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print('Modello:', percorso)
    !wget -q --show-progress -c -O '{dest}' '{url}'

In [ ]:
# === 6. AVVIO COMFYUI + LINK PUBBLICO ===
import subprocess, re, time, socket, threading

# Chiude eventuali avvii precedenti (se riesegui la cella)
subprocess.run(['pkill', '-f', 'main.py --listen'])
subprocess.run(['pkill', '-f', 'cloudflared'])

# Installa cloudflared (crea il link pubblico verso ComfyUI)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# Avvia ComfyUI in background (log in /content/comfyui.log)
log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188',
     '--output-directory', OUTPUT_DIR],
    cwd=COMFY, stdout=log, stderr=subprocess.STDOUT)

print('Avvio ComfyUI (la prima volta può richiedere qualche minuto)...')
while True:
    try:
        socket.create_connection(('127.0.0.1', 8188), timeout=1).close()
        break
    except OSError:
        if comfy.poll() is not None:
            !tail -n 40 /content/comfyui.log
            raise RuntimeError('ComfyUI si è chiuso: vedi il log qui sopra.')
        time.sleep(2)
print('ComfyUI pronto. Apro il tunnel...')

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

link = None
for riga in tunnel.stdout:
    m = re.search(r'https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com', riga)
    if m:
        link = m.group(0)
        print('\n==============================================')
        print('  APRI QUESTO LINK NEL BROWSER:')
        print('  ' + link)
        print('==============================================\n')
        break

if link is None:
    comfy.terminate()
    raise RuntimeError('cloudflared non ha creato il link. Riesegui questa cella.')

# Continua a leggere l'output del tunnel per non bloccarlo
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print('Lascia questa cella in esecuzione. Per spegnere ComfyUI, interrompi la cella.')
try:
    while comfy.poll() is None:
        time.sleep(5)
    !tail -n 40 /content/comfyui.log
    print('ComfyUI si è fermato.')
finally:
    tunnel.terminate()
    comfy.terminate()